In [3]:
import pandas as pd

# 1. Verbindung
client = MongoClient("mongodb://localhost:27017/")
db = client.Espana
collection = db.aeropuerto_stats

# 2. Daten abrufen
results = list(collection.find())

# 3. DataFrame erstellen
df = pd.DataFrame(results)

# 4. Über die Zeilen iterieren
# In Pandas nutzt man oft iterrows(), um jede Zeile einzeln zu prüfen
for index, row in df.iterrows():
    name = row['aeropuerto']
    anzahl = row['pasajero'] # Hier Singular, wie in deiner DB!
    
    if anzahl > 1000000: # Ich habe es mal auf 1 Mio erhöht für die "Großen"
        print(f"Der Flughafen {name} ist ein Gigant mit {anzahl} Passagieren.")
    elif anzahl > 10000:
        print(f"Der Flughafen {name} ist ein regionaler Knotenpunkt ({anzahl} Passagiere).")
    else:
        print(f"Der Flughafen {name} ist eher klein oder privat ({anzahl} Passagiere).")

            

Der Flughafen Adolfo Suárez Madrid-Barajas ist ein Gigant mit 68179054 Passagieren.
Der Flughafen Madrid Cuatro Vientos ist eher klein oder privat (2312 Passagiere).
Der Flughafen Barcelona-El Prat J.T. ist ein Gigant mit 57483036 Passagieren.
Der Flughafen Reus ist ein Gigant mit 1336826 Passagieren.
Der Flughafen Girona ist ein Gigant mit 2185561 Passagieren.
Der Flughafen Málaga-Costa del Sol ist ein Gigant mit 26760549 Passagieren.
Der Flughafen Alicante ist ein Gigant mit 19950394 Passagieren.
Der Flughafen Gran Canaria ist ein Gigant mit 15826553 Passagieren.
Der Flughafen Lanzarote ist ein Gigant mit 8920901 Passagieren.
Der Flughafen Tenerife Norte ist ein Gigant mit 7174977 Passagieren.
Der Flughafen Fuerteventura ist ein Gigant mit 6886935 Passagieren.
Der Flughafen La Palma ist ein Gigant mit 1533355 Passagieren.
Der Flughafen Logroño ist ein regionaler Knotenpunkt (22617 Passagiere).
Der Flughafen Tenerife Sur ist ein Gigant mit 13969678 Passagieren.
Der Flughafen El Hierro

In [4]:
import pandas as pd
from pymongo import MongoClient

# 1. Daten aus MongoDB holen
client = MongoClient("mongodb://localhost:27017/")
db = client.Espana
collection = db.aeropuerto_stats
df = pd.DataFrame(list(collection.find({"jahr": 2025})))

# 2. Den Gesamtverkehr in Spanien berechnen
total_pasajeros_espana = df['pasajero'].sum()

# 3. Den prozentualen Anteil berechnen (neue Spalte)
# Wir runden auf 2 Nachkommastellen für die Lesbarkeit im Buch
df['anteil_prozent'] = (df['pasajero'] / total_pasajeros_espana) * 100

# 4. Nach Anteil sortieren (höchste zuerst)
df_sorted = df.sort_values(by='anteil_prozent', ascending=False)

# 5. Ergebnis ausgeben (Top 10)
print(f"Gesamtpassagiere Spanien 2025: {total_pasajeros_espana:,}")
print("-" * 50)
print(df_sorted[['aeropuerto', 'comunidad', 'pasajero', 'anteil_prozent']].head(10))

# 6. Speziell für dein Andalusien-Projekt:
andalucia_share = df[df['comunidad'] == 'Andalucia']['anteil_prozent'].sum()
print("-" * 50)
print(f"Andalusien wickelt {andalucia_share:.2f}% des gesamten spanischen Flugverkehrs ab.")

Gesamtpassagiere Spanien 2025: 349,697,340
--------------------------------------------------
                      aeropuerto             comunidad  pasajero  \
0   Adolfo Suárez Madrid-Barajas                Madrid  68179054   
2         Barcelona-El Prat J.T.              Cataluña  57483036   
19             Palma de Mallorca        Islas Baleares  33806427   
44        Santiago de Compostela               Galicia  31207569   
5           Málaga-Costa del Sol             Andalucia  26760549   
6                       Alicante  Comunidad Valenciana  19950394   
7                   Gran Canaria              Canarias  15826553   
13                  Tenerife Sur              Canarias  13969678   
17                      Valencia  Comunidad Valenciana  11847527   
26                       Sevilla             Andalucia   9686487   

    anteil_prozent  
0        19.496589  
2        16.437939  
19        9.667339  
44        8.924165  
5         7.652489  
6         5.705046  
7         

In [5]:
import pandas as pd
from pymongo import MongoClient

# 1. Daten aus MongoDB holen
client = MongoClient("mongodb://localhost:27017/")
db = client.Espana
collection = db.aeropuerto_stats

# Wir holen alle Daten für 2025
df = pd.DataFrame(list(collection.find({"jahr": 2025})))

# 2. Berechnungen
total_pasajeros = df['pasajero'].sum()
df['anteil_prozent'] = (df['pasajero'] / total_pasajeros) * 100

# Nach Anteil sortieren
df_sorted = df.sort_values(by='anteil_prozent', ascending=False)

# 3. Export als CSV
df_sorted.to_csv("spanien_flughafen_analyse_2025.csv", index=False, sep=',', encoding='utf-8')

# 4. Export als HTML (schön formatiert für deinen Browser)
html_content = f"""
<html>
<head>
    <title>Analyse spanischer Flughafen-Traffic 2025</title>
    <style>
        body {{ font-family: Arial, sans-serif; margin: 20px; }}
        table {{ border-collapse: collapse; width: 100%; }}
        th, td {{ border: 1px solid #ddd; padding: 8px; text-align: left; }}
        th {{ background-color: #f2f2f2; }}
        tr:nth-child(even) {{ background-color: #f9f9f9; }}
        .highlight {{ font-weight: bold; color: #d35400; }}
    </style>
</head>
<body>
    <h1>Flughafen-Statistik Spanien 2025</h1>
    <p>Gesamtpassagiere (Aena Daten): <strong>{total_pasajeros:,}</strong></p>
    {df_sorted[['aeropuerto', 'comunidad', 'pasajero', 'anteil_prozent']].to_html(index=False, classes='table')}
    <p><i>Generiert von Sven's Data-Bot</i></p>
</body>
</html>
"""

with open("analyse_flughafen.html", "w", encoding="utf-8") as f:
    f.write(html_content)

print("HTML- und CSV-Dateien wurden erfolgreich erstellt!")

HTML- und CSV-Dateien wurden erfolgreich erstellt!
